In [1]:
"""
매핑된 재료의 ingredient_prices 행 추가 스크립트
- map_ingredient_prices.py로 avg_price는 복사됐지만
  ingredient_prices 테이블엔 없는 재료들을 기준 재료의 가격 행을 복사해서 추가
"""

import mysql.connector
from mysql.connector import Error
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("sync_prices.log", encoding="utf-8")
    ]
)
log = logging.getLogger(__name__)

DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "database": "cooking_db",
    "user": "root",
    "password": "root",
    "charset": "utf8mb4"
}


def main():
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor(dictionary=True)
        log.info("DB 연결 성공")
    except Error as e:
        log.error(f"DB 연결 실패: {e}")
        return

    try:
        # 1. avg_price는 있는데 ingredient_prices엔 없는 재료 찾기
        cursor.execute("""
            SELECT i.id, i.name, i.avg_price
            FROM ingredients i
            WHERE i.avg_price IS NOT NULL
            AND i.id NOT IN (SELECT DISTINCT ingredient_id FROM ingredient_prices)
        """)
        missing = cursor.fetchall()
        log.info(f"ingredient_prices에 없는 재료: {len(missing)}개")

        inserted = 0
        skipped = 0

        for ing in missing:
            ing_id    = ing['id']
            ing_name  = ing['name']
            avg_price = float(ing['avg_price'])

            # 2. avg_price가 같은 기준 재료의 ingredient_prices 행 찾기
            cursor.execute("""
                SELECT ip.*
                FROM ingredient_prices ip
                JOIN ingredients i ON ip.ingredient_id = i.id
                WHERE i.avg_price = %s
                LIMIT 1
            """, (avg_price,))
            ref = cursor.fetchone()

            if not ref:
                log.warning(f"기준 행 없음: {ing_name} ({avg_price}원)")
                skipped += 1
                continue

            # 3. 기준 재료의 가격 행을 복사해서 현재 재료에 삽입
            cursor.execute("""
                INSERT INTO ingredient_prices
                    (ingredient_id, currency, price_per_piece, price_per_100g, price_per_100ml,
                     source, region, effective_date)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    price_per_piece  = VALUES(price_per_piece),
                    price_per_100g   = VALUES(price_per_100g),
                    price_per_100ml  = VALUES(price_per_100ml)
            """, (
                ing_id,
                ref['currency'],
                ref['price_per_piece'],
                ref['price_per_100g'],
                ref['price_per_100ml'],
                ref['source'],
                ref['region'],
                ref['effective_date'],
            ))
            log.info(f"  추가: {ing_name} (piece={ref['price_per_piece']}, 100g={ref['price_per_100g']})")
            inserted += 1

        conn.commit()
        log.info(f"=== 완료 ===")
        log.info(f"추가: {inserted}개 / 스킵: {skipped}개")

        # 결과 확인
        cursor.execute("SELECT COUNT(*) as cnt FROM ingredient_prices")
        row = cursor.fetchone()
        log.info(f"ingredient_prices 총 {row['cnt']}개")

    except Error as e:
        log.error(f"DB 오류: {e}")
        conn.rollback()
    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    main()

2026-03-08 18:04:20,956 [INFO] DB 연결 성공
2026-03-08 18:04:20,962 [INFO] ingredient_prices에 없는 재료: 380개
2026-03-08 18:04:20,966 [INFO]   추가: LA갈비 (piece=None, 100g=15730.00)
2026-03-08 18:04:20,968 [INFO]   추가: 가다랑어 (piece=None, 100g=2827.00)
2026-03-08 18:04:20,971 [INFO]   추가: 가락국수 (piece=None, 100g=394.44)
2026-03-08 18:04:20,976 [INFO]   추가: 가락국수면 (piece=None, 100g=394.44)
2026-03-08 18:04:20,978 [INFO]   추가: 가래떡 (piece=None, 100g=555.63)
2026-03-08 18:04:20,982 [INFO]   추가: 간 돼지고기 (piece=None, 100g=3180.00)
2026-03-08 18:04:20,985 [INFO]   추가: 간돼지고기 (piece=None, 100g=3180.00)
2026-03-08 18:04:20,987 [INFO]   추가: 간소고기 (piece=None, 100g=15730.00)
2026-03-08 18:04:20,992 [INFO]   추가: 갈비 (piece=None, 100g=15730.00)
2026-03-08 18:04:20,995 [INFO]   추가: 갈비탕 (piece=None, 100g=15730.00)
2026-03-08 18:04:21,000 [INFO]   추가: 감자전분 (piece=None, 100g=548.00)
2026-03-08 18:04:21,004 [INFO]   추가: 강력분 (piece=None, 100g=171.00)
2026-03-08 18:04:21,007 [INFO]   추가: 건 고추 (piece=None, 100g=2651.50)
202